# **Notebook 4: Model Prediction (Inferences)**
> ### **Project: PestNeuroVision**
>**Description:** This notebook loads the trained model generated in Notebook 1 to perform inferences on the test set; it also generates feature maps of the predictions made.
>
> ---
>**Dependency:** Uses the trained model (`best.pt`) and specific samples from the test set.
>
> ---
>**License:** This notebook is licensed under the [GNU Affero General Public License v3.0 (AGPL-3.0)](https://www.gnu.org/licenses/agpl-3.0). See details in the [GitHub repository](https://github.com/alexander-lm/PestNeuroVision/tree/main).
>
>**Third-party software:** This project uses YOLO11 by [Ultralytics](https://github.com/ultralytics/ultralytics), distributed under the [GNU Affero General Public License v3.0](https://github.com/ultralytics/ultralytics/blob/main/LICENSE).

In [ ]:
!pip install Ultralytics

In [ ]:
import glob
import shutil
import os
from IPython.display import Image, display
from ultralytics import settings, YOLO

In [ ]:
if os.path.exists("weights"):
    shutil.rmtree("weights")
    print("Folder weights deleted.")

if os.path.exists("inference_dataset"):
    shutil.rmtree("inference_dataset")
    print("Folder inference_dataset deleted.")

if os.path.exists('runs/detect/predict'):
    shutil.rmtree('runs/detect/predict')
    print("Previous predict deleted.")

# Download the trained model (PesNeuroVision) and the test dataset from GitHub
!git init PestNeuroVision
%cd PestNeuroVision
!git remote add origin https://github.com/alexander-lm/PestNeuroVision.git
!git sparse-checkout init --cone
!git sparse-checkout set model-results/weights "dataset/inference_dataset_(CC BY 4.0)" "dataset/inference_dataset_(CC0 1.0)" image_credits.csv
!git pull --depth 1 origin main
!mv model-results/weights /content/weights
!mv "dataset/inference_dataset_(CC BY 4.0)" /content/inference_dataset
!find "dataset/inference_dataset_(CC0 1.0)" -type f -exec cp {} /content/inference_dataset/ \;
!mv image_credits.csv /content/image_credits.csv
%cd /content
!rm -rf PestNeuroVision


###**Execution of inferences**

In [ ]:
# Trained model (PestNeuroVision)
model = YOLO('/content/weights/best.pt') # Trained model (PestNeuroVision)

# Test dataset
source = sorted(glob.glob('/content/inference_dataset/*'))

# Model prediction
class_name = model.names

results = model.predict(source=source, save=True, visualize=False, verbose=False)

# Note: You can see the images with the detections made in the following folder: /content/runs/detect/predict*
for result in results:
    bbox_list = result.boxes.xywhn.tolist()
    clss_list = result.boxes.cls.int().tolist()
    conf_list = result.boxes.conf.tolist()
    for box, cls, conf in zip(bbox_list, clss_list, conf_list):
        print(f'Bounding box: {box}, Class index: {cls}, Class name: {class_name[cls]}, Confidence: {conf}')

for image_path in glob.glob(f'runs/detect/predict/*.jpg'):
  display(Image(filename=image_path, height=400))
  print('\n')